In [1]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import os

In [2]:
file_path = "data/IMDB_Dataset.csv"
df = pd.read_csv(file_path)
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [3]:
# import re
# from bs4 import BeautifulSoup



# def clean(df):
    
#     def preprocess(text):
#         text = BeautifulSoup(text, "html.parser").get_text()
#         text = text.lower()
#         text = text.replace("'", "")
#         text = re.sub(r"[^\w\s]", " ", text)
#         text = re.sub(r"\s+", " ", text).strip()
#         return text

#     df["review"] = df["review"].astype(str).apply(preprocess)
#     return df


# df = clean(df)
# df.head()


In [4]:
import re
from bs4 import BeautifulSoup

stop_words = ['ourselves','hers','between','yourself','but','again','there','about','once','during',
              'out','very','having','with','they','own','an','be','some','for','do','its','yours',
              'such','into','of','most','itself','other','off','is','s','am','or','who','as','from',
              'him','each','the','themselves','until','below','are','we','these','your','his',
              'through','me','were','her','more','himself','this','down','should','our','their',
              'while','above','both','up','to','ours','had','she','all','when','at','any','before',
              'them','same','and','been','have','in','will','on','does','yourselves','then','that',
              'because','what','over','why','so','can','did','now','under','he','you','herself',
              'has','just','where','too','only','myself','which','those','i','after','few','whom',
              't','being','if','theirs','my','against','a','by','doing','it','how','further','was',
              'here','than']

def clean(df):
    def preprocess(text):
        text = BeautifulSoup(text, "html.parser").get_text()
        text = text.lower()
        text = text.replace("'", "")
        text = re.sub(r"[^\w\s]", " ", text)
        tokens = text.split()
        tokens = [w for w in tokens if w not in stop_words]
        text = " ".join(tokens)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    
    df["review"] = df["review"].astype(str).apply(preprocess)
    return df

df = clean(df)
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [5]:
labels = df.sentiment.tolist()
documents = df.review.tolist()
print(documents[:5])
print(labels[:5])

['one reviewers mentioned watching 1 oz episode youll hooked right exactly happened first thing struck oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pulls no punches regards drugs sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy not high agenda em city home many aryans muslims gangstas latinos christians italians irish scuffles death stares dodgy dealings shady agreements never far away would say main appeal show due fact goes shows wouldnt dare forget pretty pictures painted mainstream audiences forget charm forget romance oz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high levels graphic violence not violence injustice crooked guards wholl sold nickel inmates wholl kill order get away well mannered middl

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    documents, labels, test_size=0.3, random_state=42
)

In [7]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)


In [8]:
model = MultinomialNB()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
probs = model.predict_proba(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8582


In [9]:
print(predictions[:5])
print(probs[:5])

['positive' 'positive' 'negative' 'positive' 'negative']
[[4.15431038e-01 5.84568962e-01]
 [9.47016525e-10 9.99999999e-01]
 [9.99999999e-01 1.04444726e-09]
 [3.77912724e-07 9.99999622e-01]
 [9.99972009e-01 2.79909598e-05]]


In [ ]:
results_df = pd.DataFrame({
    "review": X_test_raw,   
    "true_label": y_test,
    "predicted_label": predictions,
    "prob_negative": probs[:, 0],
    "prob_positive": probs[:, 1],
})

results_df

,review,true_label,predicted_label,prob_negative,prob_positive
0,really liked summerslam due look arena curtain...,positive,positive,4.154310e-01,5.845690e-01
1,not many television shows appeal quite many di...,positive,positive,9.470165e-10,1.000000e+00
2,film quickly gets major chase scene ever incre...,negative,negative,1.000000e+00,1.044447e-09
3,jane austen would definitely approve one gwyne...,positive,positive,3.779127e-07,9.999996e-01
4,expectations somewhat high went see movie thou...,negative,negative,9.999720e-01,2.799096e-05
...,...,...,...,...,...
14995,landscape battle opens escaping prisoners snow...,positive,positive,2.985436e-35,1.000000e+00
14996,jake speed 1986 amusing parody indiana jones a...,positive,positive,1.430373e-04,9.998570e-01
14997,plan b appearance quickly made unedited sloppy...,negative,negative,9.994073e-01,5.926977e-04
14998,one perks job things slow watch movie downstai...,positive,positive,4.987427e-04,9.995013e-01


In [12]:
new_review = ["this movie was good and i enjoyed watching it"]
v = vectorizer.transform(new_review)
new_review_pred_label = model.predict(v)[0]
new_review_probs = model.predict_proba(v)[0]
print(f"Review: {new_review[0]}")
print(f"Predicted label: {new_review_pred_label}")
print(f"neg: {new_review_probs[0]:.4f}, pos: {new_review_probs[1]:.4f}")


Review: this movie was good and i enjoyed watching it
Predicted label: positive
neg: 0.4454, pos: 0.5546
